In [ ]:
# Example 1: Basic GraphQL Schema and Object Types
import graphene

# Define a basic GraphQL object type
class User(graphene.ObjectType):
    id = graphene.ID()
    name = graphene.String()
    email = graphene.String()
    age = graphene.Int()
    is_active = graphene.Boolean()

# Define another object type with relationships
class Post(graphene.ObjectType):
    id = graphene.ID()
    title = graphene.String()
    content = graphene.String()
    author = graphene.Field(User)
    created_at = graphene.String()

print("Basic GraphQL Schema Creation:")

# Sample data
users_data = [
    {"id": "1", "name": "Alice", "email": "alice@example.com", "age": 30, "is_active": True},
    {"id": "2", "name": "Bob", "email": "bob@example.com", "age": 25, "is_active": False},
    {"id": "3", "name": "Charlie", "email": "charlie@example.com", "age": 35, "is_active": True}
]

posts_data = [
    {"id": "1", "title": "GraphQL Basics", "content": "Learning GraphQL with Python", "author_id": "1", "created_at": "2023-09-15"},
    {"id": "2", "title": "Python Tips", "content": "Advanced Python techniques", "author_id": "2", "created_at": "2023-09-14"},
    {"id": "3", "title": "API Design", "content": "RESTful vs GraphQL APIs", "author_id": "1", "created_at": "2023-09-13"}
]

# Define query root
class Query(graphene.ObjectType):
    user = graphene.Field(User, id=graphene.ID(required=True))
    all_users = graphene.List(User)
    post = graphene.Field(Post, id=graphene.ID(required=True))
    all_posts = graphene.List(Post)

    def resolve_user(self, info, id):
        user_data = next((u for u in users_data if u["id"] == id), None)
        return User(**user_data) if user_data else None

    def resolve_all_users(self, info):
        return [User(**user) for user in users_data]

    def resolve_post(self, info, id):
        post_data = next((p for p in posts_data if p["id"] == id), None)
        if post_data:
            author_data = next((u for u in users_data if u["id"] == post_data["author_id"]), None)
            return Post(
                id=post_data["id"],
                title=post_data["title"],
                content=post_data["content"],
                author=User(**author_data) if author_data else None,
                created_at=post_data["created_at"]
            )
        return None

    def resolve_all_posts(self, info):
        posts = []
        for post_data in posts_data:
            author_data = next((u for u in users_data if u["id"] == post_data["author_id"]), None)
            posts.append(Post(
                id=post_data["id"],
                title=post_data["title"],
                content=post_data["content"],
                author=User(**author_data) if author_data else None,
                created_at=post_data["created_at"]
            ))
        return posts

# Create schema
schema = graphene.Schema(query=Query)

# Test queries
print("Testing basic queries:")
result = schema.execute("{ allUsers { id name email } }")
print(f"All users: {result.data}")

result = schema.execute("{ user(id: \"1\") { name email age } }")
print(f"Single user: {result.data}")

result = schema.execute("{ allPosts { title author { name } } }")
print(f"Posts with authors: {result.data}")

print("✅ Basic GraphQL schema and object types completed")

## Example 2: Mutations and Input Types

In [ ]:
# Example 2: Mutations and Input Types
import graphene

print("GraphQL Mutations and Input Types:")

# Input types for mutations
class UserInput(graphene.InputObjectType):
    name = graphene.String(required=True)
    email = graphene.String(required=True)
    age = graphene.Int()

class PostInput(graphene.InputObjectType):
    title = graphene.String(required=True)
    content = graphene.String(required=True)
    author_id = graphene.ID(required=True)

# Mutations
class CreateUser(graphene.Mutation):
    class Arguments:
        user_data = UserInput(required=True)

    user = graphene.Field(User)
    success = graphene.Boolean()
    message = graphene.String()

    def mutate(self, info, user_data):
        # Generate new ID
        new_id = str(len(users_data) + 1)
        
        # Create new user
        new_user_data = {
            "id": new_id,
            "name": user_data.name,
            "email": user_data.email,
            "age": user_data.age or 0,
            "is_active": True
        }
        
        # Add to data store
        users_data.append(new_user_data)
        
        return CreateUser(
            user=User(**new_user_data),
            success=True,
            message=f"User {user_data.name} created successfully"
        )

class UpdateUser(graphene.Mutation):
    class Arguments:
        id = graphene.ID(required=True)
        user_data = UserInput(required=True)

    user = graphene.Field(User)
    success = graphene.Boolean()
    message = graphene.String()

    def mutate(self, info, id, user_data):
        # Find user to update
        user_index = next((i for i, u in enumerate(users_data) if u["id"] == id), None)
        
        if user_index is None:
            return UpdateUser(
                user=None,
                success=False,
                message="User not found"
            )
        
        # Update user data
        users_data[user_index].update({
            "name": user_data.name,
            "email": user_data.email,
            "age": user_data.age or users_data[user_index]["age"]
        })
        
        return UpdateUser(
            user=User(**users_data[user_index]),
            success=True,
            message=f"User {user_data.name} updated successfully"
        )

class CreatePost(graphene.Mutation):
    class Arguments:
        post_data = PostInput(required=True)

    post = graphene.Field(Post)
    success = graphene.Boolean()
    message = graphene.String()

    def mutate(self, info, post_data):
        # Check if author exists
        author_exists = any(u["id"] == post_data.author_id for u in users_data)
        if not author_exists:
            return CreatePost(
                post=None,
                success=False,
                message="Author not found"
            )
        
        # Generate new post ID
        new_id = str(len(posts_data) + 1)
        
        # Create new post
        new_post_data = {
            "id": new_id,
            "title": post_data.title,
            "content": post_data.content,
            "author_id": post_data.author_id,
            "created_at": "2023-09-15"
        }
        
        # Add to data store
        posts_data.append(new_post_data)
        
        # Get author for response
        author_data = next(u for u in users_data if u["id"] == post_data.author_id)
        
        return CreatePost(
            post=Post(
                id=new_id,
                title=post_data.title,
                content=post_data.content,
                author=User(**author_data),
                created_at="2023-09-15"
            ),
            success=True,
            message=f"Post '{post_data.title}' created successfully"
        )

# Define Mutation root
class Mutation(graphene.ObjectType):
    create_user = CreateUser.Field()
    update_user = UpdateUser.Field()
    create_post = CreatePost.Field()

# Create schema with mutations
schema_with_mutations = graphene.Schema(query=Query, mutation=Mutation)

# Test mutations
print("Testing mutations:")

# Create a new user
create_user_query = '''
mutation {
  createUser(userData: {name: "Diana", email: "diana@example.com", age: 28}) {
    user {
      id
      name
      email
      age
    }
    success
    message
  }
}
'''

result = schema_with_mutations.execute(create_user_query)
print(f"Create user result: {result.data}")

# Update an existing user
update_user_query = '''
mutation {
  updateUser(id: "1", userData: {name: "Alice Smith", email: "alice.smith@example.com", age: 31}) {
    user {
      id
      name
      email
      age
    }
    success
    message
  }
}
'''

result = schema_with_mutations.execute(update_user_query)
print(f"Update user result: {result.data}")

# Create a new post
create_post_query = '''
mutation {
  createPost(postData: {title: "GraphQL Mutations", content: "How to handle mutations in GraphQL", authorId: "1"}) {
    post {
      id
      title
      author {
        name
      }
    }
    success
    message
  }
}
'''

result = schema_with_mutations.execute(create_post_query)
print(f"Create post result: {result.data}")

print("✅ Mutations and input types completed")

## Example 3: Advanced Field Types and Scalars

In [ ]:
# Example 3: Advanced Field Types and Scalars
import graphene
from graphene import Scalar
from graphql.language import ast
import json
from datetime import datetime

print("Advanced Field Types and Custom Scalars:")

# Custom scalar type for JSON
class JSONScalar(Scalar):
    """JSON scalar type"""
    
    @staticmethod
    def serialize(dt):
        return json.dumps(dt)
    
    @staticmethod
    def parse_literal(node):
        if isinstance(node, ast.StringValue):
            return json.loads(node.value)
    
    @staticmethod
    def parse_value(value):
        return json.loads(value) if isinstance(value, str) else value

# Custom scalar for DateTime
class DateTime(Scalar):
    """DateTime scalar type"""
    
    @staticmethod
    def serialize(dt):
        return dt.isoformat() if isinstance(dt, datetime) else dt
    
    @staticmethod
    def parse_literal(node):
        if isinstance(node, ast.StringValue):
            return datetime.fromisoformat(node.value)
    
    @staticmethod
    def parse_value(value):
        return datetime.fromisoformat(value) if isinstance(value, str) else value

# Enum type
class UserRole(graphene.Enum):
    ADMIN = "admin"
    MODERATOR = "moderator"
    USER = "user"
    GUEST = "guest"

class PostStatus(graphene.Enum):
    DRAFT = "draft"
    PUBLISHED = "published"
    ARCHIVED = "archived"

# Advanced object types with custom scalars
class AdvancedUser(graphene.ObjectType):
    id = graphene.ID()
    name = graphene.String()
    email = graphene.String()
    role = graphene.Field(UserRole)
    metadata = graphene.Field(JSONScalar)
    created_at = graphene.Field(DateTime)
    last_login = graphene.Field(DateTime)
    
    # Computed field
    display_name = graphene.String()
    
    def resolve_display_name(self, info):
        role_title = self.role.title() if self.role else "User"
        return f"{self.name} ({role_title})"

class AdvancedPost(graphene.ObjectType):
    id = graphene.ID()
    title = graphene.String()
    content = graphene.String()
    status = graphene.Field(PostStatus)
    author = graphene.Field(AdvancedUser)
    tags = graphene.List(graphene.String)
    view_count = graphene.Int()
    metadata = graphene.Field(JSONScalar)
    created_at = graphene.Field(DateTime)
    updated_at = graphene.Field(DateTime)
    
    # Computed fields
    is_published = graphene.Boolean()
    word_count = graphene.Int()
    
    def resolve_is_published(self, info):
        return self.status == PostStatus.PUBLISHED
    
    def resolve_word_count(self, info):
        return len(self.content.split()) if self.content else 0

# Advanced data with custom types
advanced_users_data = [
    {
        "id": "1",
        "name": "Alice",
        "email": "alice@example.com",
        "role": UserRole.ADMIN,
        "metadata": {"department": "Engineering", "team": "Backend"},
        "created_at": datetime(2023, 1, 15),
        "last_login": datetime(2023, 9, 15, 10, 30)
    },
    {
        "id": "2",
        "name": "Bob",
        "email": "bob@example.com",
        "role": UserRole.MODERATOR,
        "metadata": {"department": "Content", "team": "Editorial"},
        "created_at": datetime(2023, 2, 20),
        "last_login": datetime(2023, 9, 14, 15, 45)
    }
]

advanced_posts_data = [
    {
        "id": "1",
        "title": "Advanced GraphQL Patterns",
        "content": "This post covers advanced GraphQL patterns including custom scalars, unions, and interfaces.",
        "status": PostStatus.PUBLISHED,
        "author_id": "1",
        "tags": ["graphql", "python", "advanced"],
        "view_count": 1250,
        "metadata": {"featured": True, "reading_time": 15},
        "created_at": datetime(2023, 9, 10),
        "updated_at": datetime(2023, 9, 12)
    },
    {
        "id": "2",
        "title": "Draft Article",
        "content": "This is a draft article about GraphQL best practices.",
        "status": PostStatus.DRAFT,
        "author_id": "2",
        "tags": ["graphql", "best-practices"],
        "view_count": 0,
        "metadata": {"draft_version": 3},
        "created_at": datetime(2023, 9, 13),
        "updated_at": datetime(2023, 9, 15)
    }
]

# Advanced query with filtering
class AdvancedQuery(graphene.ObjectType):
    advanced_user = graphene.Field(AdvancedUser, id=graphene.ID(required=True))
    advanced_users = graphene.List(AdvancedUser, role=graphene.Argument(UserRole))
    advanced_post = graphene.Field(AdvancedPost, id=graphene.ID(required=True))
    advanced_posts = graphene.List(
        AdvancedPost, 
        status=graphene.Argument(PostStatus),
        tag=graphene.String(),
        min_views=graphene.Int()
    )
    
    def resolve_advanced_user(self, info, id):
        user_data = next((u for u in advanced_users_data if u["id"] == id), None)
        return AdvancedUser(**user_data) if user_data else None
    
    def resolve_advanced_users(self, info, role=None):
        users = advanced_users_data
        if role:
            users = [u for u in users if u["role"] == role]
        return [AdvancedUser(**user) for user in users]
    
    def resolve_advanced_post(self, info, id):
        post_data = next((p for p in advanced_posts_data if p["id"] == id), None)
        if post_data:
            author_data = next((u for u in advanced_users_data if u["id"] == post_data["author_id"]), None)
            post_dict = dict(post_data)
            post_dict["author"] = AdvancedUser(**author_data) if author_data else None
            return AdvancedPost(**post_dict)
        return None
    
    def resolve_advanced_posts(self, info, status=None, tag=None, min_views=None):
        posts = advanced_posts_data
        
        # Filter by status
        if status:
            posts = [p for p in posts if p["status"] == status]
        
        # Filter by tag
        if tag:
            posts = [p for p in posts if tag in p["tags"]]
        
        # Filter by minimum views
        if min_views is not None:
            posts = [p for p in posts if p["view_count"] >= min_views]
        
        # Convert to objects
        result = []
        for post_data in posts:
            author_data = next((u for u in advanced_users_data if u["id"] == post_data["author_id"]), None)
            post_dict = dict(post_data)
            post_dict["author"] = AdvancedUser(**author_data) if author_data else None
            result.append(AdvancedPost(**post_dict))
        
        return result

# Create advanced schema
advanced_schema = graphene.Schema(query=AdvancedQuery)

# Test advanced queries
print("Testing advanced field types:")

# Query with custom scalars and enums
result = advanced_schema.execute('''
{
  advancedUsers {
    id
    name
    role
    displayName
    metadata
    createdAt
  }
}
''')
print(f"Advanced users: {result.data}")

# Query with filtering
result = advanced_schema.execute('''
{
  advancedPosts(status: PUBLISHED, minViews: 1000) {
    id
    title
    status
    isPublished
    wordCount
    viewCount
    tags
    metadata
    author {
      displayName
      role
    }
  }
}
''')
print(f"Filtered posts: {result.data}")

# Query specific user
result = advanced_schema.execute('''
{
  advancedUser(id: "1") {
    name
    role
    metadata
    lastLogin
  }
}
''')
print(f"Specific user: {result.data}")

print("✅ Advanced field types and scalars completed")

## Example 4: Interfaces, Unions, and Schema Introspection

In [ ]:
# Example 4: Interfaces, Unions, and Schema Introspection
import graphene

print("Interfaces, Unions, and Schema Introspection:")

# Define interfaces
class Node(graphene.Interface):
    id = graphene.ID(required=True)

class Timestamped(graphene.Interface):
    created_at = graphene.Field(DateTime)
    updated_at = graphene.Field(DateTime)

# Implement interfaces
class Article(graphene.ObjectType):
    class Meta:
        interfaces = (Node, Timestamped)
    
    title = graphene.String()
    content = graphene.String()
    author_name = graphene.String()
    category = graphene.String()

class Video(graphene.ObjectType):
    class Meta:
        interfaces = (Node, Timestamped)
    
    title = graphene.String()
    duration = graphene.Int()  # in seconds
    url = graphene.String()
    thumbnail = graphene.String()

class Podcast(graphene.ObjectType):
    class Meta:
        interfaces = (Node, Timestamped)
    
    title = graphene.String()
    duration = graphene.Int()  # in seconds
    audio_url = graphene.String()
    episode_number = graphene.Int()

# Union type for different content types
class Content(graphene.Union):
    class Meta:
        types = (Article, Video, Podcast)

# Notification types
class CommentNotification(graphene.ObjectType):
    id = graphene.ID()
    message = graphene.String()
    post_id = graphene.ID()
    commenter_name = graphene.String()

class LikeNotification(graphene.ObjectType):
    id = graphene.ID()
    message = graphene.String()
    post_id = graphene.ID()
    liker_name = graphene.String()

class SystemNotification(graphene.ObjectType):
    id = graphene.ID()
    message = graphene.String()
    system_type = graphene.String()

# Union for notifications
class Notification(graphene.Union):
    class Meta:
        types = (CommentNotification, LikeNotification, SystemNotification)

# Sample data for interfaces and unions
content_data = [
    {
        "type": "article",
        "id": "1",
        "title": "GraphQL Interfaces",
        "content": "Understanding GraphQL interfaces and unions",
        "author_name": "Alice",
        "category": "Technology",
        "created_at": datetime(2023, 9, 1),
        "updated_at": datetime(2023, 9, 5)
    },
    {
        "type": "video",
        "id": "2",
        "title": "GraphQL Tutorial",
        "duration": 1800,  # 30 minutes
        "url": "https://example.com/video/graphql-tutorial",
        "thumbnail": "https://example.com/thumbnails/graphql.jpg",
        "created_at": datetime(2023, 9, 10),
        "updated_at": datetime(2023, 9, 10)
    },
    {
        "type": "podcast",
        "id": "3",
        "title": "GraphQL Deep Dive",
        "duration": 3600,  # 1 hour
        "audio_url": "https://example.com/podcast/graphql-deep-dive.mp3",
        "episode_number": 42,
        "created_at": datetime(2023, 9, 12),
        "updated_at": datetime(2023, 9, 12)
    }
]

notifications_data = [
    {
        "type": "comment",
        "id": "1",
        "message": "New comment on your post",
        "post_id": "123",
        "commenter_name": "Bob"
    },
    {
        "type": "like",
        "id": "2",
        "message": "Someone liked your post",
        "post_id": "456",
        "liker_name": "Charlie"
    },
    {
        "type": "system",
        "id": "3",
        "message": "System maintenance scheduled",
        "system_type": "maintenance"
    }
]

# Query with interfaces and unions
class InterfaceUnionQuery(graphene.ObjectType):
    # Query all content (returns union)
    all_content = graphene.List(Content)
    
    # Query by interface
    all_nodes = graphene.List(Node)
    all_timestamped = graphene.List(Timestamped)
    
    # Query notifications (union)
    notifications = graphene.List(Notification)
    
    # Schema introspection
    schema_types = graphene.List(graphene.String)
    
    def resolve_all_content(self, info):
        content = []
        for item in content_data:
            if item["type"] == "article":
                content.append(Article(**item))
            elif item["type"] == "video":
                content.append(Video(**item))
            elif item["type"] == "podcast":
                content.append(Podcast(**item))
        return content
    
    def resolve_all_nodes(self, info):
        # Return all objects that implement Node interface
        nodes = []
        for item in content_data:
            if item["type"] == "article":
                nodes.append(Article(**item))
            elif item["type"] == "video":
                nodes.append(Video(**item))
            elif item["type"] == "podcast":
                nodes.append(Podcast(**item))
        return nodes
    
    def resolve_all_timestamped(self, info):
        # Return all objects that implement Timestamped interface
        timestamped = []
        for item in content_data:
            if item["type"] == "article":
                timestamped.append(Article(**item))
            elif item["type"] == "video":
                timestamped.append(Video(**item))
            elif item["type"] == "podcast":
                timestamped.append(Podcast(**item))
        return timestamped
    
    def resolve_notifications(self, info):
        notifications = []
        for notif in notifications_data:
            if notif["type"] == "comment":
                notifications.append(CommentNotification(**notif))
            elif notif["type"] == "like":
                notifications.append(LikeNotification(**notif))
            elif notif["type"] == "system":
                notifications.append(SystemNotification(**notif))
        return notifications
    
    def resolve_schema_types(self, info):
        # Return available types in schema
        return ["Article", "Video", "Podcast", "CommentNotification", "LikeNotification", "SystemNotification"]

# Create schema with interfaces and unions
interface_schema = graphene.Schema(query=InterfaceUnionQuery)

# Test interface and union queries
print("Testing interfaces and unions:")

# Query union types with fragments
union_query = '''
{
  allContent {
    ... on Article {
      id
      title
      content
      authorName
      category
    }
    ... on Video {
      id
      title
      duration
      url
      thumbnail
    }
    ... on Podcast {
      id
      title
      duration
      audioUrl
      episodeNumber
    }
  }
}
'''

result = interface_schema.execute(union_query)
print(f"Union query result: {result.data}")

# Query interface fields
interface_query = '''
{
  allNodes {
    id
    ... on Timestamped {
      createdAt
      updatedAt
    }
  }
}
'''

result = interface_schema.execute(interface_query)
print(f"Interface query result: {result.data}")

# Query notifications union
notifications_query = '''
{
  notifications {
    ... on CommentNotification {
      id
      message
      postId
      commenterName
    }
    ... on LikeNotification {
      id
      message
      postId
      likerName
    }
    ... on SystemNotification {
      id
      message
      systemType
    }
  }
}
'''

result = interface_schema.execute(notifications_query)
print(f"Notifications query result: {result.data}")

# Schema introspection
print("\nSchema Introspection:")
introspection_query = '''
{
  __schema {
    types {
      name
      kind
    }
  }
}
'''

result = interface_schema.execute(introspection_query)
type_names = [t['name'] for t in result.data['__schema']['types'] if not t['name'].startswith('__')]
print(f"Schema types: {type_names[:10]}")  # Show first 10

# Type introspection
type_introspection_query = '''
{
  __type(name: "Article") {
    name
    fields {
      name
      type {
        name
      }
    }
    interfaces {
      name
    }
  }
}
'''

result = interface_schema.execute(type_introspection_query)
print(f"Article type introspection: {result.data}")

print("✅ Interfaces, unions, and schema introspection completed")

## Example 5: Production GraphQL API with Error Handling and Middleware

In [ ]:
# Example 5: Production GraphQL API with Error Handling and Middleware
import graphene
from graphene import ObjectType
import time

print("Production GraphQL API with Error Handling and Middleware:")

# Custom exception classes
class ValidationError(Exception):
    def __init__(self, message, field=None):
        self.message = message
        self.field = field
        super().__init__(message)

class AuthenticationError(Exception):
    def __init__(self, message="Authentication required"):
        self.message = message
        super().__init__(message)

class AuthorizationError(Exception):
    def __init__(self, message="Insufficient permissions"):
        self.message = message
        super().__init__(message)

# Middleware for logging and authentication
class LoggingMiddleware:
    def resolve(self, next, root, info, **args):
        start_time = time.time()
        field_name = info.field_name
        
        print(f"→ Executing field: {field_name}")
        
        try:
            result = next(root, info, **args)
            execution_time = time.time() - start_time
            print(f"✓ Field {field_name} completed in {execution_time:.3f}s")
            return result
        except Exception as e:
            execution_time = time.time() - start_time
            print(f"✗ Field {field_name} failed in {execution_time:.3f}s: {e}")
            raise

class AuthenticationMiddleware:
    def resolve(self, next, root, info, **args):
        # Simulate checking for auth token in context
        context = info.context or {}
        user_id = context.get('user_id')
        
        # Only check auth for mutations and sensitive queries
        if info.operation.operation == 'mutation' or 'admin' in info.field_name.lower():
            if not user_id:
                raise AuthenticationError("Authentication required for this operation")
        
        # Add user info to context
        if user_id:
            context['authenticated'] = True
        
        return next(root, info, **args)

# Production-ready user type with validation
class ProductionUser(graphene.ObjectType):
    id = graphene.ID()
    username = graphene.String()
    email = graphene.String()
    full_name = graphene.String()
    is_active = graphene.Boolean()
    role = graphene.String()
    created_at = graphene.Field(DateTime)
    
    def resolve_email(self, info):
        # Hide email for non-authenticated users
        context = info.context or {}
        if not context.get('authenticated'):
            return None
        return self.email

# Error handling types
class FieldError(graphene.ObjectType):
    field = graphene.String()
    message = graphene.String()

class MutationResult(graphene.ObjectType):
    success = graphene.Boolean()
    errors = graphene.List(FieldError)
    message = graphene.String()

# Production mutations with comprehensive error handling
class CreateUserMutation(graphene.Mutation):
    class Arguments:
        username = graphene.String(required=True)
        email = graphene.String(required=True)
        full_name = graphene.String(required=True)
        password = graphene.String(required=True)

    user = graphene.Field(ProductionUser)
    result = graphene.Field(MutationResult)

    def mutate(self, info, username, email, full_name, password):
        errors = []
        
        # Validation
        if len(username) < 3:
            errors.append(FieldError(field="username", message="Username must be at least 3 characters"))
        
        if '@' not in email:
            errors.append(FieldError(field="email", message="Invalid email format"))
        
        if len(password) < 8:
            errors.append(FieldError(field="password", message="Password must be at least 8 characters"))
        
        # Check if username already exists
        existing_users = [u for u in users_data if u.get('username') == username]
        if existing_users:
            errors.append(FieldError(field="username", message="Username already exists"))
        
        if errors:
            return CreateUserMutation(
                user=None,
                result=MutationResult(
                    success=False,
                    errors=errors,
                    message="Validation failed"
                )
            )
        
        # Create user
        new_user = {
            "id": str(len(users_data) + 100),
            "username": username,
            "email": email,
            "full_name": full_name,
            "is_active": True,
            "role": "user",
            "created_at": datetime.now()
        }
        
        users_data.append(new_user)
        
        return CreateUserMutation(
            user=ProductionUser(**new_user),
            result=MutationResult(
                success=True,
                errors=[],
                message=f"User {username} created successfully"
            )
        )

# Admin-only queries and mutations
class AdminQuery(graphene.ObjectType):
    admin_users = graphene.List(ProductionUser)
    system_stats = graphene.Field(graphene.String)
    
    def resolve_admin_users(self, info):
        # This will trigger authentication middleware
        return [ProductionUser(**user) for user in users_data]
    
    def resolve_system_stats(self, info):
        return f"Total users: {len(users_data)}, Active users: {len([u for u in users_data if u.get('is_active', True)])}"

# Rate limiting simulation
class RateLimitingMiddleware:
    def __init__(self):
        self.request_counts = {}
    
    def resolve(self, next, root, info, **args):
        # Simulate IP-based rate limiting
        context = info.context or {}
        client_ip = context.get('client_ip', '127.0.0.1')
        
        current_time = time.time()
        minute_bucket = int(current_time // 60)
        
        key = f"{client_ip}:{minute_bucket}"
        self.request_counts[key] = self.request_counts.get(key, 0) + 1
        
        if self.request_counts[key] > 100:  # 100 requests per minute
            raise Exception("Rate limit exceeded. Please try again later.")
        
        return next(root, info, **args)

# Production query with comprehensive features
class ProductionQuery(graphene.ObjectType):
    user = graphene.Field(ProductionUser, id=graphene.ID(required=True))
    users = graphene.List(ProductionUser, limit=graphene.Int(default_value=10))
    search_users = graphene.List(ProductionUser, query=graphene.String(required=True))
    
    # Admin-only fields
    admin_users = graphene.Field(AdminQuery)
    
    def resolve_user(self, info, id):
        user_data = next((u for u in users_data if u["id"] == id), None)
        if not user_data:
            raise Exception(f"User with id {id} not found")
        return ProductionUser(**user_data)
    
    def resolve_users(self, info, limit):
        if limit > 100:
            raise ValidationError("Limit cannot exceed 100", field="limit")
        
        return [ProductionUser(**user) for user in users_data[:limit]]
    
    def resolve_search_users(self, info, query):
        if len(query) < 2:
            raise ValidationError("Search query must be at least 2 characters", field="query")
        
        filtered_users = [
            user for user in users_data 
            if query.lower() in user.get('username', '').lower() or 
               query.lower() in user.get('full_name', '').lower()
        ]
        
        return [ProductionUser(**user) for user in filtered_users]
    
    def resolve_admin_users(self, info):
        return AdminQuery()

# Production mutation root
class ProductionMutation(graphene.ObjectType):
    create_user = CreateUserMutation.Field()

# Create production schema with middleware
production_schema = graphene.Schema(
    query=ProductionQuery,
    mutation=ProductionMutation
)

# Test production features
print("Testing production GraphQL API:")

# Test with authentication context
auth_context = {"user_id": "1", "client_ip": "192.168.1.1"}

# Test user query
result = production_schema.execute(
    'query { user(id: "1") { id username email fullName } }',
    context=auth_context,
    middleware=[LoggingMiddleware(), AuthenticationMiddleware()]
)
print(f"Authenticated user query: {result.data}")

# Test without authentication
no_auth_context = {"client_ip": "192.168.1.1"}

result = production_schema.execute(
    'query { user(id: "1") { id username email fullName } }',
    context=no_auth_context,
    middleware=[LoggingMiddleware(), AuthenticationMiddleware()]
)
print(f"Non-authenticated user query (email hidden): {result.data}")

# Test validation errors
create_invalid_user = '''
mutation {
  createUser(username: "ab", email: "invalid", fullName: "Test User", password: "123") {
    user {
      id
      username
    }
    result {
      success
      errors {
        field
        message
      }
      message
    }
  }
}
'''

result = production_schema.execute(
    create_invalid_user,
    context=auth_context,
    middleware=[LoggingMiddleware(), AuthenticationMiddleware()]
)
print(f"Validation error result: {result.data}")

# Test search with rate limiting
try:
    rate_limiter = RateLimitingMiddleware()
    
    result = production_schema.execute(
        'query { searchUsers(query: "alice") { username fullName } }',
        context=auth_context,
        middleware=[LoggingMiddleware(), rate_limiter]
    )
    print(f"Search result: {result.data}")
    
except Exception as e:
    print(f"Rate limiting or other error: {e}")
    raise

# Test error handling
try:
    result = production_schema.execute(
        'query { user(id: "999") { username } }',
        context=auth_context,
        middleware=[LoggingMiddleware()]
    )
    if result.errors:
        print(f"GraphQL errors: {[str(error) for error in result.errors]}")
except Exception as e:
    print(f"Execution error: {e}")
    raise

print("✅ Production GraphQL API with error handling and middleware completed")